In [2]:
import requests
import time

OLAS_MECH_SUBGRAPH_URL = (
    "https://api.subgraph.staging.autonolas.tech/api/proxy/mech-marketplace-gnosis"
)
QUESTION_DATA_SEPARATOR = "\u241f"

GET_MECH_SENDER_QUERY = """
query MechSender($id: ID!, $timestamp_gt: Int!, $skip: Int, $first: Int) {
    sender(id: $id) {
        totalMarketplaceRequests
        requests(first: $first, skip: $skip, where: { blockTimestamp_gt: $timestamp_gt }) {
            parsedRequest {
                questionTitle
                tool
                prompt
            }
        }
    }
}
"""


def fetch_all_mech_requests(
    agent_address: str, timestamp_gt: int = None, batch_size: int = 1000
):
    """
    Fetch all mech requests for a given agent address from the subgraph, using the exact query and batching logic from production.
    Returns a list of mech requests.
    Only fetches requests from the last 3 days by default.
    """
    if timestamp_gt is None:
        # Default: last 3 days
        timestamp_gt = int(time.time()) - 3 * 24 * 60 * 60
    all_requests = []
    skip = 0
    while True:
        variables = {
            "id": agent_address,
            "timestamp_gt": timestamp_gt,
            "skip": skip,
            "first": batch_size,
        }
        response = requests.post(
            OLAS_MECH_SUBGRAPH_URL,
            json={"query": GET_MECH_SENDER_QUERY, "variables": variables},
            headers={"Content-Type": "application/json"},
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        print(f"dat{data=}")
        result = data.get("data", {}).get("sender", {})
        batch_requests = result.get("requests", [])
        if not batch_requests:
            break
        all_requests.extend(batch_requests)
        if len(batch_requests) < batch_size:
            break
        skip += batch_size
    return all_requests


def extract_question_title(question: str) -> str:
    """
    Extracts the question title using the separator logic from production code.
    """
    if not question:
        return ""
    return question.split(QUESTION_DATA_SEPARATOR)[0]


def match_bet_to_mech_request(bet: dict, mech_requests: list):
    """
    Given a bet dict and a list of mech requests, match the mech request by question title.
    Uses the same separator logic as production.
    Returns the matching mech request(s).
    """
    question = bet.get("question", "")
    bet_title = extract_question_title(question)
    if not bet_title:
        return []
    matches = [
        req
        for req in mech_requests
        if extract_question_title(
            (req.get("parsedRequest", {}) or {}).get("questionTitle", "")
        )
        == bet_title
    ]
    return matches


# Example usage:
if __name__ == "__main__":
    agent_address = "0xcf282a37bae6f505fa51443631005bb6032322bd"
    mech_requests = fetch_all_mech_requests(agent_address)
    # Example bet dict
    bet = {
        "question": "Will the International Commission on Zoological Nomenclature (ICZN) officially recognize and list Spinosaurus mirabilis as a valid species in its online registry on or before February 25, 2026?"
    }
    matched_requests = match_bet_to_mech_request(bet, mech_requests)
    print(matched_requests)

datdata={'errors': [{'message': 'indexing_error'}]}
[]


In [ ]:
for i in matched_requests:
    print(json.dumps(i, indent=2))

In [ ]:
import requests


def fetch_last_bets(n=10):
    url = "https://api.subgraph.staging.autonolas.tech/api/proxy/predict-omen"
    headers = {"Content-Type": "application/json"}

    query = f"""
    {{
      bets(
        first: {n}
        orderBy: timestamp
        orderDirection: desc
        where: {{fixedProductMarketMaker_: {{currentAnswer_not: null}}}}
      ) {{
        id
        bettor {{
          id
          serviceId
        }}
        outcomeIndex
        fixedProductMarketMaker {{
          currentAnswer
          question
        }}
      }}
    }}
    """

    response = requests.post(url, headers=headers, json={"query": query})
    data = response.json()

    bets = data["data"]["bets"]

    formatted_bets = []
    for bet in bets:
        chosen = int(bet["outcomeIndex"])
        correct = int(bet["fixedProductMarketMaker"]["currentAnswer"], 16)

        bettor_address = bet["bettor"]["id"]
        service_id = bet["bettor"]["serviceId"]

        bet = {
            "bet_id": bet["id"],
            "bettor": bettor_address,
            "service_id": service_id,
            "chosen_outcome": chosen,
            "correct_outcome": correct,
            "is_correct": chosen == correct,
            "question": bet["fixedProductMarketMaker"]["question"],
        }
        formatted_bets.append(bet)
    return formatted_bets


bets = fetch_last_bets(10000)

In [4]:
import requests
PREDICT_OMEN_URL = "https://api.subgraph.staging.autonolas.tech/api/proxy/predict-omen"


def fetch_last_bets(n: int = 100, batch_size: int = 1000) -> list[dict]:
    """
    Fetch the last `n` resolved bets from the predict-omen subgraph.
    Only includes bets where the market has a current (resolved) answer.
    Paginates automatically when `n` exceeds the subgraph page limit.
    """
    headers = {"Content-Type": "application/json"}
    all_raw_bets: list[dict] = []
    skip = 0

    while len(all_raw_bets) < n:
        first = min(batch_size, n - len(all_raw_bets))
        query = f"""
    {{
      bets(
        first: {first}
        skip: {skip}
        orderBy: timestamp
        orderDirection: desc
        where: {{fixedProductMarketMaker_: {{currentAnswer_not: null}}}}
      ) {{
        id
        timestamp
        bettor {{
          id
          serviceId
        }}
        outcomeIndex
        fixedProductMarketMaker {{
          currentAnswer
          question
        }}
      }}
    }}
    """
        response = requests.post(
            PREDICT_OMEN_URL, headers=headers, json={"query": query}, timeout=30
        )
        response.raise_for_status()
        data = response.json()
        batch = data["data"]["bets"]
        if not batch:
            break
        all_raw_bets.extend(batch)
        if len(batch) < first:
            break
        skip += first

    formatted_bets = []
    for bet in all_raw_bets:
        chosen = int(bet["outcomeIndex"])
        correct = int(bet["fixedProductMarketMaker"]["currentAnswer"], 16)
        formatted_bets.append(
            {
                "bet_id": bet["id"],
                "timestamp": int(bet["timestamp"]),
                "bettor": bet["bettor"]["id"],
                "service_id": bet["bettor"]["serviceId"],
                "chosen_outcome": chosen,
                "correct_outcome": correct,
                "is_correct": chosen == correct,
                "question": bet["fixedProductMarketMaker"]["question"],
            }
        )
    return formatted_bets


In [9]:
from collections import Counter

bets = fetch_last_bets(10000)

chosen_dist = Counter(b["chosen_outcome"] for b in bets)
correct_dist = Counter(b["correct_outcome"] for b in bets)
correctness_dist = Counter(b["is_correct"] for b in bets)

print(f"Total bets: {len(bets)}")
print()
print("Chosen outcome distribution:")
for outcome, count in sorted(chosen_dist.items()):
    print(f"  Outcome {outcome}: {count} ({count / len(bets) * 100:.1f}%)")
print()
print("Correct outcome distribution:")
for outcome, count in sorted(correct_dist.items()):
    print(f"  Outcome {outcome}: {count} ({count / len(bets) * 100:.1f}%)")
print()
print("Correctness distribution:")
for is_correct, count in sorted(correctness_dist.items()):
    label = "Correct" if is_correct else "Incorrect"
    print(f"  {label}: {count} ({count / len(bets) * 100:.1f}%)")


Total bets: 10000

Chosen outcome distribution:
  Outcome 0: 5784 (57.8%)
  Outcome 1: 4216 (42.2%)

Correct outcome distribution:
  Outcome 0: 1010 (10.1%)
  Outcome 1: 8893 (88.9%)
  Outcome 115792089237316195423570985008687907853269984665640564039457584007913129639935: 97 (1.0%)

Correctness distribution:
  Incorrect: 5273 (52.7%)
  Correct: 4727 (47.3%)
